In [ ]:
import scipy.signal.signaltools

def _centered(arr, newsize):
    # Return the center newsize portion of the array.
    newsize = np.asarray(newsize)
    currsize = np.array(arr.shape)
    startind = (currsize - newsize) // 2
    endind = startind + newsize
    myslice = [slice(startind[k], endind[k]) for k in range(len(endind))]
    return arr[tuple(myslice)]

scipy.signal.signaltools._centered = _centered

import eventstudy as es
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import pandas_datareader as web

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/dfTimeComplate.pkl")
#("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/df_dates.pkl")

In [ ]:
df = df[["VIX return","Negative return"]]
df.dropna(inplace=True)
df = df[(df.T != 0).any()]
df = df.loc[df["Negative return"]!=0]
df = df.loc[df["VIX return"]!=0]
df_dates = df
df_dates["Time"] = df_dates.index
df_dates['Time']=df_dates['Time'].dt.strftime('%Y-%m-%d %H:%M:%S')
df_dates.index = df_dates['Time']

In [ ]:
df_dates

In [ ]:
events_db = []
for date, row in df_dates.T.iteritems():
    print(date)
    new_row = {'event_date': np.datetime64(row["Time"]),'market_ticker':'VIX return','security_ticker': 'Negative return'}#row["Company"]}
    events_db.append(new_row)

In [ ]:
events_db

In [ ]:
returns = df_dates[["Negative return","VIX return"]]
returns.index=df_dates.Time

In [ ]:
len(df_dates)

In [ ]:
len(events_db)

In [ ]:
returns.index.names = ['date']

In [ ]:
returns.to_csv("returns_list.csv")

In [ ]:
returns

In [ ]:
es.Single.import_returns("./CSV/returns.csv")
# es.Single.import_FamaFrench("FamaFrenchSPY.csv")

In [ ]:
tickers={"Negative return"}

In [ ]:
events=events_db

In [ ]:
# tickers = ['SPY']
releases = dict()
for ticker in tickers:
    try:
        events = [event for event in events_db if event['security_ticker']==ticker]
        print(ticker)
        releases[ticker] = es.Multiple.from_list(
            events,
            es.Single.market_model,#FamaFrench_3factor,
            event_window= (-5,+15)
            # estimation_size=(-10000,-5)
        )
        print("##########################")
    except:
        print("EXCEPTION: "+ticker)

In [ ]:
events

In [ ]:
len(events)

In [ ]:
releases.values

In [ ]:
ticker

In [ ]:
for ticker, event in releases.items():
    event.plot()
    print(ticker)
    # plt.figure(figsize=(20, 20))
    plt.xlabel('Event Day')
    plt.title('Negative Return and VIX Return Event Study')
    plt.show()